In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

BASE_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == "plots" else Path(r"C:\Users\eduque\Documents\models\sectoral-energy-demands")
RESIDENTIAL_FILE = BASE_DIR / "residential" / "output_data" / "final_energy_demand_all_end_uses.csv"
TRANSPORT_DIR = BASE_DIR / "transport" / "output_data"
INDUSTRY_FILE = BASE_DIR / "industry" / "output_data" / "final_energy_demand_detailed_sectors_wide.csv"

print("Using base dir:", BASE_DIR)
print("Residential:", RESIDENTIAL_FILE.exists())
print("Transport dir:", TRANSPORT_DIR.exists())
print("Industry:", INDUSTRY_FILE.exists())

Using base dir: C:\Users\eduque\Documents\Models\sectoral-energy-demands
Residential: True
Transport dir: True
Industry: True


In [2]:
def _normalize_scenario_name(raw: str) -> str:
    if raw is None or pd.isna(raw):
        return "Unknown"
    text = str(raw).strip()
    lowered = text.lower()
    mapping = {
        "bau": "BAU",
        "progressive": "Progressive",
        "optimistic": "Optimistic",
        "historic": "Historic",
        "base": "Historic",
        "baseline": "Historic",
    }
    return mapping.get(lowered, text)


def _parse_transport_scenario_year(value) -> tuple[str, int]:
    text = str(value).strip()
    m = re.match(r"^(\d{4})_(.+)$", text)
    if m:
        return _normalize_scenario_name(m.group(2)), int(m.group(1))
    if text.isdigit() and len(text) == 4:
        return "Historic", int(text)
    # Conservative fallback: keep scenario text and unknown year marker
    return _normalize_scenario_name(text), -1


def load_residential_totals(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, keep_default_na=False, na_values=[""])
    out = (
        df.groupby(["country", "scenario", "year"], as_index=False)["final_energy_demand_twh"]
        .sum()
        .rename(columns={"final_energy_demand_twh": "total_twh"})
    )
    out["scenario"] = out["scenario"].map(_normalize_scenario_name)
    out["sector"] = "Residential"
    return out[["country", "scenario", "year", "sector", "total_twh"]]


def load_transport_totals(path_dir: Path) -> pd.DataFrame:
    files = sorted(path_dir.glob("transport*.csv"))
    if not files:
        return pd.DataFrame(columns=["country", "scenario", "year", "sector", "total_twh"])

    chunks = []
    for file in files:
        tmp = pd.read_csv(file)
        num_cols = [c for c in tmp.columns if c not in ["scenario", "country"]]
        tmp["total_twh"] = tmp[num_cols].fillna(0).sum(axis=1)
        parsed = tmp["scenario"].map(_parse_transport_scenario_year)
        tmp["scenario"] = parsed.map(lambda t: t[0])
        tmp["year"] = parsed.map(lambda t: t[1])
        chunks.append(tmp[["country", "scenario", "year", "total_twh"]])

    out = pd.concat(chunks, ignore_index=True)
    out = out[out["year"] > 0].copy()
    out = out.groupby(["country", "scenario", "year"], as_index=False)["total_twh"].sum()
    out["sector"] = "Transport"
    return out[["country", "scenario", "year", "sector", "total_twh"]]


def load_industry_totals(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, keep_default_na=False, na_values=[""])
    id_cols = ["country", "scenario", "year", "carrier"]
    sector_cols = [c for c in df.columns if c not in id_cols]
    melted = df.melt(id_vars=id_cols, value_vars=sector_cols, value_name="value_mwh", var_name="industry_subsector")
    out = (
        melted.groupby(["country", "scenario", "year"], as_index=False)["value_mwh"]
        .sum()
        .rename(columns={"value_mwh": "total_mwh"})
    )
    out["total_twh"] = out["total_mwh"] / 1_000_000.0
    out["scenario"] = out["scenario"].map(_normalize_scenario_name)
    out["sector"] = "Industry"
    return out[["country", "scenario", "year", "sector", "total_twh"]]


df_res = load_residential_totals(RESIDENTIAL_FILE)
df_trn = load_transport_totals(TRANSPORT_DIR)
df_ind = load_industry_totals(INDUSTRY_FILE)

df_all = pd.concat([df_res, df_trn, df_ind], ignore_index=True)
df_all["country"] = df_all["country"].astype(str).str.strip()
df_all["year"] = df_all["year"].astype(int)

print("Rows by sector:")
print(df_all.groupby("sector").size())
print("Scenarios:", sorted(df_all["scenario"].unique()))
print("Years:", sorted(df_all["year"].unique()))

Rows by sector:
sector
Industry       60
Residential    72
Transport      64
dtype: int64
Scenarios: ['BAU', 'Historic', 'Optimistic', 'Progressive']
Years: [np.int64(2020), np.int64(2030), np.int64(2035), np.int64(2040), np.int64(2045), np.int64(2050)]


In [3]:
# Report table 1: sector totals per country x scenario x year
report_sector = (
    df_all.pivot_table(
        index=["country", "scenario", "year"],
        columns="sector",
        values="total_twh",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reset_index()
 )

for col in ["Industry", "Residential", "Transport"]:
    if col not in report_sector.columns:
        report_sector[col] = 0.0

report_sector["Total"] = report_sector[["Industry", "Residential", "Transport"]].sum(axis=1)
report_sector = report_sector.sort_values(["country", "scenario", "year"]).reset_index(drop=True)

# Report table 2: shares by sector (%)
share_cols = ["Industry", "Residential", "Transport"]
report_share = report_sector[["country", "scenario", "year", *share_cols, "Total"]].copy()
for c in share_cols:
    report_share[f"{c}_share_pct"] = np.where(
        report_share["Total"] > 0,
        100.0 * report_share[c] / report_share["Total"],
        0.0,
    )

# Report table 3: scenario growth vs 2020 by country and sector
baseline = (
    report_sector[report_sector["year"] == 2020][["country", "scenario", *share_cols]]
    .rename(columns={c: f"{c}_2020" for c in share_cols})
 )
report_growth = report_sector.merge(baseline, on=["country", "scenario"], how="left")
for c in share_cols:
    report_growth[f"{c}_growth_vs_2020_pct"] = np.where(
        report_growth[f"{c}_2020"] > 0,
        100.0 * (report_growth[c] - report_growth[f"{c}_2020"]) / report_growth[f"{c}_2020"],
        np.nan,
    )

print("Top rows of sector totals report:")
display(report_sector.head(20))

print("Top rows of sector shares report (%):")
display(report_share[["country", "scenario", "year", "Industry_share_pct", "Residential_share_pct", "Transport_share_pct"]].head(20))

print("Top rows of growth report (% vs 2020):")
display(report_growth[["country", "scenario", "year", "Industry_growth_vs_2020_pct", "Residential_growth_vs_2020_pct", "Transport_growth_vs_2020_pct"]].head(20))

Top rows of sector totals report:


sector,country,scenario,year,Industry,Residential,Transport,Total
0,Egypt,BAU,2020,0.000,168.946,0.000,168.946
1,Egypt,BAU,2030,992.208,204.185,376.766,"1,573.159"
2,Egypt,BAU,2035,"1,295.842",226.135,612.758,"2,134.735"
3,Egypt,BAU,2040,"1,900.507",208.203,942.391,"3,051.101"
4,Egypt,BAU,2045,"3,166.659",228.424,"1,381.525","4,776.609"
5,Egypt,BAU,2050,"5,896.861",249.801,"1,924.408","8,071.071"
6,Egypt,Historic,2020,0.000,0.000,128.241,128.241
7,Egypt,Optimistic,2020,0.000,172.815,0.000,172.815
8,Egypt,Optimistic,2030,"1,221.140",166.714,371.618,"1,759.471"
9,Egypt,Optimistic,2035,"2,172.195",169.697,600.605,"2,942.497"


Top rows of sector shares report (%):


sector,country,scenario,year,Industry_share_pct,Residential_share_pct,Transport_share_pct
0,Egypt,BAU,2020,0.000,100.000,0.000
1,Egypt,BAU,2030,63.071,12.979,23.950
2,Egypt,BAU,2035,60.703,10.593,28.704
3,Egypt,BAU,2040,62.289,6.824,30.887
4,Egypt,BAU,2045,66.295,4.782,28.923
5,Egypt,BAU,2050,73.062,3.095,23.843
6,Egypt,Historic,2020,0.000,0.000,100.000
7,Egypt,Optimistic,2020,0.000,100.000,0.000
8,Egypt,Optimistic,2030,69.404,9.475,21.121
9,Egypt,Optimistic,2035,73.821,5.767,20.411


Top rows of growth report (% vs 2020):


sector,country,scenario,year,Industry_growth_vs_2020_pct,Residential_growth_vs_2020_pct,Transport_growth_vs_2020_pct
0,Egypt,BAU,2020,NaN,0.000,NaN
1,Egypt,BAU,2030,NaN,20.858,NaN
2,Egypt,BAU,2035,NaN,33.850,NaN
3,Egypt,BAU,2040,NaN,23.236,NaN
4,Egypt,BAU,2045,NaN,35.205,NaN
5,Egypt,BAU,2050,NaN,47.859,NaN
6,Egypt,Historic,2020,NaN,NaN,0.000
7,Egypt,Optimistic,2020,NaN,0.000,NaN
8,Egypt,Optimistic,2030,NaN,-3.531,NaN
9,Egypt,Optimistic,2035,NaN,-1.804,NaN


In [4]:
# Critical data checks before plotting layer
issues = []

if (report_sector["Total"] < 0).any():
    issues.append("Negative total demand found.")

for c in ["Industry", "Residential", "Transport"]:
    if (report_sector[c] < 0).any():
        issues.append(f"Negative values found in {c}.")

missing_2020 = report_growth[(report_growth["year"] > 2020) & report_growth[["Industry_2020", "Residential_2020", "Transport_2020"]].isna().any(axis=1)]
if not missing_2020.empty:
    issues.append(f"Missing 2020 baseline for {len(missing_2020)} country-scenario rows.")

print("Data quality status:")
if issues:
    for item in issues:
        print("-", item)
else:
    print("- No critical issues detected for summary reporting.")

print("\nSuggested next graphics (from this harmonized dataset):")
suggestions = [
    "1) Country dashboard: one page per country with sector totals over time by scenario.",
    "2) Sector share trajectories: stacked area of sector shares (Industry/Transport/Residential) by scenario.",
    "3) Scenario delta chart: Progressive and Optimistic minus BAU for each year and country.",
    "4) Growth heatmap: % growth vs 2020 across country x year for each sector.",
    "5) Consistency panel: compare sum-of-sectors totals against original sector outputs as QA chart.",
]
for s in suggestions:
    print(s)

Data quality status:
- No critical issues detected for summary reporting.

Suggested next graphics (from this harmonized dataset):
1) Country dashboard: one page per country with sector totals over time by scenario.
2) Sector share trajectories: stacked area of sector shares (Industry/Transport/Residential) by scenario.
3) Scenario delta chart: Progressive and Optimistic minus BAU for each year and country.
4) Growth heatmap: % growth vs 2020 across country x year for each sector.
5) Consistency panel: compare sum-of-sectors totals against original sector outputs as QA chart.
